# Download Common Voice Kaggle dataset

Run this notebook in Google Colab. It only downloads the dataset and prints a compact structure preview. After that, send the output back so the next cell can be written against the actual files.

In [1]:
!pip -q install kagglehub

In [2]:
import os
from pathlib import Path

import kagglehub

DATASET = "prateeknarain/common-voice"

# Download latest version
dataset_path = Path(kagglehub.dataset_download(DATASET))

print("Path to dataset files:", dataset_path)
print("Exists:", dataset_path.exists())
print("Absolute path:", dataset_path.resolve())

100%|██████████| 15.6G/15.6G [03:13<00:00, 86.5MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/prateeknarain/common-voice/versions/1
Exists: True
Absolute path: /root/.cache/kagglehub/datasets/prateeknarain/common-voice/versions/1


In [3]:
from collections import Counter

def print_tree(root: Path, max_depth: int = 3, max_items_per_dir: int = 40):
    root = Path(root)
    print(root)

    def walk(path: Path, depth: int):
        if depth > max_depth:
            return

        try:
            entries = sorted(path.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
        except PermissionError:
            print("  " * depth + "[permission denied]")
            return

        shown = entries[:max_items_per_dir]
        hidden_count = max(0, len(entries) - len(shown))

        for entry in shown:
            prefix = "  " * depth + "- "
            if entry.is_dir():
                print(prefix + entry.name + "/")
                walk(entry, depth + 1)
            else:
                size_mb = entry.stat().st_size / (1024 * 1024)
                print(prefix + f"{entry.name} ({size_mb:.2f} MB)")

        if hidden_count:
            print("  " * depth + f"... {hidden_count} more items")

    walk(root, 1)

print_tree(dataset_path, max_depth=3, max_items_per_dir=40)

/root/.cache/kagglehub/datasets/prateeknarain/common-voice/versions/1
  - Common Voice/
    - Common Voice/
      - English/
      - German/
      - Hindi/
      - Japanese/
      - Russian/
      - Spanish/
  - DS_10283_3443/
    - VCTK-Corpus-0.92/
      - txt/
      - wav48_silence_trimmed/
      - speaker-info.txt (0.00 MB)
      - update.txt (0.00 MB)
    - README.txt (0.00 MB)


In [4]:
all_files = [p for p in dataset_path.rglob('*') if p.is_file()]
suffix_counts = Counter(p.suffix.lower() or '[no extension]' for p in all_files)

print('Total files:', len(all_files))
print('File extensions:')
for suffix, count in suffix_counts.most_common(30):
    print(f'  {suffix}: {count}')

metadata_files = [
    p for p in all_files
    if p.suffix.lower() in {'.csv', '.tsv', '.json'}
]
print('\nMetadata files (.csv/.tsv/.json):', len(metadata_files))
for p in sorted(metadata_files, key=lambda path: (path.suffix, str(path)))[:120]:
    print('-', p.relative_to(dataset_path), f'({p.stat().st_size / (1024 * 1024):.2f} MB)')
if len(metadata_files) > 120:
    print(f'... {len(metadata_files) - 120} more metadata files')


Total files: 271143
File extensions:
  .mp3: 138144
  .flac: 88328
  .txt: 44587
  .tsv: 84

Metadata files (.csv/.tsv/.json): 84
- Common Voice/Common Voice/English/cv-corpus-21.0-delta-2025-03-14-en/14765303714 14765303717/cv-corpus-21.0-delta-2025-03-14/en/clip_durations.tsv (0.71 MB)
- Common Voice/Common Voice/English/cv-corpus-21.0-delta-2025-03-14-en/14765303730 14765303730/cv-corpus-21.0-delta-2025-03-14/en/invalidated.tsv (0.01 MB)
- Common Voice/Common Voice/English/cv-corpus-21.0-delta-2025-03-14-en/14765303730 14765303730/cv-corpus-21.0-delta-2025-03-14/en/other.tsv (6.69 MB)
- Common Voice/Common Voice/English/cv-corpus-21.0-delta-2025-03-14-en/14765303730 14765303730/cv-corpus-21.0-delta-2025-03-14/en/validated.tsv (0.08 MB)
- Common Voice/Common Voice/English/cv-corpus-21.0-delta-2025-03-14-en/14765303731 14765303731/cv-corpus-21.0-delta-2025-03-14/en/reported.tsv (0.02 MB)
- Common Voice/Common Voice/English/cv-corpus-21.0-delta-2025-03-14-en/14765303731 14765304003/cv-

## Build normalized English dataset

Creates `dataset/wav` and `dataset/metadata.csv` from the English Common Voice manifest. The Kaggle cache stays unchanged. Set `MAX_ITEMS = None` for the full selected manifest, or keep a small number for a quick smoke export.

In [5]:
import csv
import shutil
import subprocess
from pathlib import Path

import pandas as pd

# Output layout:
#   /content/dataset/wav/<clip_stem>.wav
#   /content/dataset/metadata.csv
OUTPUT_ROOT = Path('/content/dataset')
WAV_DIR = OUTPUT_ROOT / 'wav'
METADATA_CSV = OUTPUT_ROOT / 'metadata.csv'

# validated.tsv alone has only 249 English clips in this Kaggle delta.
# Include other.tsv to build a usable English Common Voice export from this archive.
MANIFEST_NAMES = ['validated.tsv', 'other.tsv']
MAX_ITEMS = None
SAMPLE_RATE = 16_000

english_tsvs = sorted(
    [
        p for p in dataset_path.rglob('*.tsv')
        if '/English/' in p.as_posix() or '/en/' in p.as_posix()
    ],
    key=lambda p: (p.name, len(p.parts), str(p)),
)
assert english_tsvs, 'No English TSV files found under dataset_path'

# Prefer canonical paths and avoid duplicated shard paths like ...-en/<range>/cv-corpus.../en/*.tsv.
def is_canonical(path: Path) -> bool:
    rel = path.relative_to(dataset_path).as_posix()
    return '/cv-corpus-21.0-delta-2025-03-14/en/' in rel and '-en/' not in rel

canonical_tsvs = [p for p in english_tsvs if is_canonical(p)] or english_tsvs
print('English TSV summary:')
for p in canonical_tsvs:
    try:
        rows_count = sum(1 for _ in p.open('r', encoding='utf-8')) - 1
    except Exception as exc:
        rows_count = f'error: {exc}'
    print(f'- {p.relative_to(dataset_path)} | {p.stat().st_size / (1024 * 1024):.2f} MB | rows={rows_count}')

manifest_paths = [p for p in canonical_tsvs if p.name in MANIFEST_NAMES]
assert manifest_paths, f'No selected manifests found: {MANIFEST_NAMES}'
print()
print('Selected manifests:')
for p in manifest_paths:
    print('-', p.relative_to(dataset_path))

tables = []
for manifest_path in manifest_paths:
    table_part = pd.read_csv(manifest_path, sep='\t')
    required_columns = {'path', 'sentence'}
    missing_columns = required_columns - set(table_part.columns)
    if missing_columns:
        print(f'Skipping {manifest_path.name}: missing columns {sorted(missing_columns)}')
        continue
    table_part = table_part.dropna(subset=['path', 'sentence']).copy()
    table_part['path'] = table_part['path'].astype(str)
    table_part['sentence'] = table_part['sentence'].astype(str).str.strip()
    table_part = table_part[table_part['sentence'].str.len() > 0]
    table_part['_manifest_path'] = str(manifest_path)
    tables.append(table_part)

assert tables, 'No selected manifest rows with path+sentence'
table = pd.concat(tables, ignore_index=True)
table = table.drop_duplicates(subset=['path', 'sentence']).reset_index(drop=True)
if MAX_ITEMS is not None:
    table = table.head(MAX_ITEMS).reset_index(drop=True)
print()
print('Rows selected for export:', len(table))

if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
WAV_DIR.mkdir(parents=True, exist_ok=True)

def resolve_audio(row_path: str, manifest_path: Path) -> Path:
    relative = Path(row_path)
    candidates = [
        manifest_path.parent / relative,
        manifest_path.parent / 'clips' / relative.name,
        dataset_path / relative,
        dataset_path / 'clips' / relative.name,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    matches = list(dataset_path.rglob(relative.name))
    if matches:
        return matches[0]
    raise FileNotFoundError(f'Could not find audio for {row_path}')

rows = []
failed = []
for index, row in table.iterrows():
    manifest_path = Path(row['_manifest_path'])
    source_audio = resolve_audio(row['path'], manifest_path)
    output_name = f'{source_audio.stem}.wav'
    output_path = WAV_DIR / output_name

    cmd = [
        'ffmpeg', '-hide_banner', '-loglevel', 'error', '-y',
        '-i', str(source_audio),
        '-ac', '1', '-ar', str(SAMPLE_RATE),
        str(output_path),
    ]
    try:
        subprocess.run(cmd, check=True)
    except Exception as exc:
        failed.append((row['path'], str(exc)))
        continue

    rows.append({
        'file_name': f'wav/{output_name}',
        'text': row['sentence'],
        'source_manifest': manifest_path.name,
        'source_path': str(source_audio.relative_to(dataset_path)),
    })

metadata = pd.DataFrame(rows)
metadata.to_csv(METADATA_CSV, index=False, quoting=csv.QUOTE_MINIMAL)

print()
print('Output root:', OUTPUT_ROOT)
print('WAV files:', len(rows))
print('Metadata:', METADATA_CSV)
print('Failed:', len(failed))
print(metadata.head())
if failed[:5]:
    print('First failures:')
    for item in failed[:5]:
        print(' -', item)


English TSV summary:
- Common Voice/Common Voice/English/cv-corpus-21.0-delta-2025-03-14/en/clip_durations.tsv | 0.71 MB | rows=21796
- Common Voice/Common Voice/English/cv-corpus-21.0-delta-2025-03-14/en/invalidated.tsv | 0.01 MB | rows=39
- Common Voice/Common Voice/English/cv-corpus-21.0-delta-2025-03-14/en/other.tsv | 6.69 MB | rows=21508
- Common Voice/Common Voice/English/cv-corpus-21.0-delta-2025-03-14/en/reported.tsv | 0.02 MB | rows=115
- Common Voice/Common Voice/English/cv-corpus-21.0-delta-2025-03-14/en/unvalidated_sentences.tsv | 5.80 MB | rows=32229
- Common Voice/Common Voice/English/cv-corpus-21.0-delta-2025-03-14/en/validated.tsv | 0.08 MB | rows=249
- Common Voice/Common Voice/English/cv-corpus-21.0-delta-2025-03-14/en/validated_sentences.tsv | 223.62 MB | rows=1678488

Selected manifests:
- Common Voice/Common Voice/English/cv-corpus-21.0-delta-2025-03-14/en/other.tsv
- Common Voice/Common Voice/English/cv-corpus-21.0-delta-2025-03-14/en/validated.tsv

Rows selected 

In [3]:
from datetime import datetime
from pathlib import Path
import csv
import subprocess

from google.colab import drive

DATASET_ROOT = Path('/content/dataset')
WAV_DIR = DATASET_ROOT / 'wav'
METADATA_CSV = DATASET_ROOT / 'metadata.csv'

assert DATASET_ROOT.exists(), f'Missing dataset directory: {DATASET_ROOT}'
assert WAV_DIR.exists(), f'Missing wav directory: {WAV_DIR}'
assert METADATA_CSV.exists(), f'Missing metadata file: {METADATA_CSV}'

wav_count = sum(1 for _ in WAV_DIR.glob('*.wav'))
with METADATA_CSV.open('r', encoding='utf-8', newline='') as handle:
    metadata = list(csv.DictReader(handle))
metadata_rows = len(metadata)
missing_files = [row['file_name'] for row in metadata if not (DATASET_ROOT / row['file_name']).exists()]

print('WAV files:', wav_count)
print('Metadata rows:', metadata_rows)
print('Missing files referenced by metadata:', len(missing_files))
if missing_files[:10]:
    print('First missing files:', missing_files[:10])

assert wav_count == metadata_rows, f'WAV count ({wav_count}) != metadata rows ({metadata_rows})'
assert not missing_files, f'Metadata references {len(missing_files)} missing WAV files'

print('Dataset size:')
subprocess.run(['du', '-sh', str(DATASET_ROOT)], check=False)

drive.mount('/content/drive', force_remount=False)

DRIVE_DIR = Path('/content/drive/MyDrive/sanday_datasets')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

stamp = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
archive_path = DRIVE_DIR / f'common_voice_en_wav_{wav_count}_{stamp}.tar'

# Fast archive: no compression. For a smaller but much slower archive, change -cf to -czf and .tar to .tar.gz.
print('Writing archive to:', archive_path)
subprocess.run(
    ['tar', '-cf', str(archive_path), '-C', '/content', 'dataset'],
    check=True,
)

print('Archive size:')
subprocess.run(['du', '-sh', str(archive_path)], check=False)
print('Done:', archive_path)


Mounted at /content/drive
Writing archive to: /content/drive/MyDrive/sanday_datasets/common_voice_en_wav_21748_20260621_112829.tar


/tmp/ipykernel_105417/790296596.py:39: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  stamp = datetime.utcnow().strftime('%Y%m%d_%H%M%S')


Archive size:
Done: /content/drive/MyDrive/sanday_datasets/common_voice_en_wav_21748_20260621_112829.tar
